# 004 · Sets

> **Syntax › Collections** · Python 3.14

A `set` is an unordered collection of **unique**, hashable items with fast membership tests and built-in set algebra. It is Python's equivalent of C#'s `HashSet<T>`, with operators instead of method calls.

**In this notebook you will learn:**

- how to create sets — and why `{}` is *not* an empty set
- adding and removing items (`add`, `remove`, `discard`)
- union, intersection, difference and symmetric difference with operators
- subset/superset tests and `frozenset`
- order-preserving de-duplication and the hashability requirement

## C# mental model

| C# | Python |
|---|---|
| `HashSet<int>` | `set[int]` (type hint only) |
| `new HashSet<int> { 1, 2 }` | `{1, 2}` |
| `new HashSet<int>()` | `set()` |
| `set.Contains(x)` | `x in s` |
| `ImmutableHashSet<T>` / `FrozenSet<T>` (.NET 8) | `frozenset` |

The big difference: C#'s `UnionWith`, `IntersectWith`, etc. **mutate** the set in place. Python's operators `|`, `&`, `-`, `^` **return a new set**; the augmented forms `|=`, `&=`, … mutate.

## Creating sets

```csharp
// C#
var primes = new HashSet<int> { 2, 3, 5, 7 };
var empty = new HashSet<string>();
var unique = new HashSet<char>("hello");
```

In [ ]:
primes = {2, 3, 5, 7}
empty = set()
unique_letters = set("hello")           # any iterable; duplicates vanish
from_list = set([3, 1, 3, 2, 1])

print(primes, empty, unique_letters, from_list)
print(len(from_list))

⚠️ **Gotcha:** `{}` is an empty **dict**, not an empty set. Braces with no elements were taken by dicts first. Always write `set()` for an empty set.

In [ ]:
print(type({}).__name__)        # dict
print(type(set()).__name__)     # set
print(type({1}).__name__)       # set — braces with elements and no colons

⚠️ **Gotcha:** sets are **unordered**. Never rely on iteration order or printed order, and you cannot index a set. (Small ints often *appear* sorted — that is an implementation detail.) `HashSet<T>` makes the same "no order" promise.

In [ ]:
s = {"pear", "apple", "fig"}
print(sorted(s))          # sort when you need a stable order

try:
    s[0]
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Adding and removing

| C# `HashSet<T>` | Python `set` |
|---|---|
| `Add(x)` (returns `bool`) | `add(x)` (returns `None`) |
| `Remove(x)` (returns `false` if missing) | `discard(x)` — silent if missing |
| *(n/a)* | `remove(x)` — raises `KeyError` if missing |
| *(n/a)* | `pop()` — remove and return an arbitrary item |
| `Clear()` | `clear()` |
| `UnionWith(xs)` | `update(xs)` or `\|=` |

In [ ]:
tags = {"python", "csharp"}

tags.add("rust")
tags.add("python")          # already present — no effect, no error
tags.update(["go", "sql"])  # add many
print(sorted(tags))

tags.discard("java")        # not present — silently ignored
tags.remove("go")
try:
    tags.remove("java")     # not present — KeyError
except KeyError as e:
    print(f"{type(e).__name__}: {e}")
print(sorted(tags))

Since `add` doesn't return whether the item was new, check membership first if you need C#'s `if (seen.Add(x))` idiom:

```csharp
// C#
if (seen.Add(item)) Console.WriteLine($"first time: {item}");
```

In [ ]:
seen = set()
for item in ["a", "b", "a", "c", "b"]:
    if item not in seen:
        seen.add(item)
        print(f"first time: {item}")

## Membership is fast

`x in some_set` is an average **O(1)** hash lookup, while `x in some_list` is an **O(n)** scan. When you test membership repeatedly against a large collection, convert it to a set first.

In [ ]:
allowed_ids = set(range(100_000))
print(99_999 in allowed_ids)     # hash lookup — fast regardless of size
print(-1 not in allowed_ids)

## Set algebra

| Operation | C# (mutates) | Python operator (new set) | Python method (new set) | In place |
|---|---|---|---|---|
| Union | `UnionWith` | `a \| b` | `a.union(b)` | `a \|= b` |
| Intersection | `IntersectWith` | `a & b` | `a.intersection(b)` | `a &= b` |
| Difference | `ExceptWith` | `a - b` | `a.difference(b)` | `a -= b` |
| Symmetric difference | `SymmetricExceptWith` | `a ^ b` | `a.symmetric_difference(b)` | `a ^= b` |

LINQ's `Union`, `Intersect` and `Except` are the non-mutating C# equivalents of the Python operators.

In [ ]:
backend = {"alice", "bob", "carol"}
frontend = {"carol", "dave", "alice", "erin"}

print(sorted(backend | frontend))   # in either team
print(sorted(backend & frontend))   # in both teams
print(sorted(backend - frontend))   # backend only
print(sorted(backend ^ frontend))   # in exactly one team
print(sorted(backend))              # unchanged — operators return new sets

The operators require **both** operands to be sets. The method forms accept **any iterable**:

In [ ]:
a = {1, 2, 3}

print(a.union([3, 4, 5]))           # list is fine for the method
try:
    a | [3, 4, 5]
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

The augmented operators mutate in place — this is the direct equivalent of `UnionWith` and friends:

```csharp
// C#
var active = new HashSet<string> { "a", "b" };
active.UnionWith(["c"]);
active.ExceptWith(["a"]);
```

In [ ]:
active = {"a", "b"}
active |= {"c"}         # UnionWith
active -= {"a"}         # ExceptWith
print(sorted(active))

## Subsets, supersets and disjoint sets

| C# | Python |
|---|---|
| `a.IsSubsetOf(b)` | `a <= b` or `a.issubset(b)` |
| `a.IsProperSubsetOf(b)` | `a < b` |
| `a.IsSupersetOf(b)` | `a >= b` or `a.issuperset(b)` |
| `a.IsProperSupersetOf(b)` | `a > b` |
| `!a.Overlaps(b)` | `a.isdisjoint(b)` |
| `a.SetEquals(b)` | `a == b` |

In [ ]:
required = {"read", "write"}
granted = {"read", "write", "admin"}

print(required <= granted)          # subset
print(required < granted)           # proper subset
print(granted >= required)          # superset
print(required.isdisjoint({"delete"}))
print({1, 2, 3} == {3, 2, 1})       # equality ignores order

## `frozenset` — the immutable set

A `frozenset` is a set that can't be changed after creation. Because it is immutable it is **hashable**, so it can be a dict key or an element of another set (a regular `set` cannot).

In .NET the closest types are `ImmutableHashSet<T>` and `FrozenSet<T>` (new in .NET 8, optimised for read-heavy lookups).

In [ ]:
vowels = frozenset("aeiou")
print("e" in vowels)
print(vowels | {"y"})                # operators still work — return a new frozenset

try:
    vowels.add("y")
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
# frozensets as dict keys: an unordered pair -> distance
distances = {
    frozenset({"A", "B"}): 5,
    frozenset({"B", "C"}): 3,
}
print(distances[frozenset({"B", "A"})])   # order doesn't matter

try:
    {{1, 2}}                              # a set of sets — not allowed
except TypeError as e:
    print(f"{type(e).__name__}: {e}")
print({frozenset({1, 2})})                # a set of frozensets — fine

## Elements must be hashable

Like dict keys, set elements must be hashable: numbers, strings, tuples of hashables, frozensets. Lists, dicts and sets can't be elements.

In [ ]:
points = {(0, 0), (1, 2), (0, 0)}
print(points)                 # tuple elements are fine; duplicate removed

try:
    {[0, 0], [1, 2]}
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## De-duplicating while keeping order

`set(items)` removes duplicates but **loses** order. Since dicts preserve insertion order, `dict.fromkeys(items)` is the idiomatic way to de-duplicate while keeping the first occurrence of each item — like LINQ's `Distinct()`, which (in practice) preserves order.

```csharp
// C#
var unique = items.Distinct().ToList();
```

In [ ]:
visits = ["home", "about", "home", "blog", "about", "contact"]

print(sorted(set(visits)))            # unique, but original order lost
print(list(dict.fromkeys(visits)))    # unique, original order kept

## Type hints

In [ ]:
admins: set[str] = {"alice"}
lookup: frozenset[int] = frozenset({1, 2, 3})
print(admins, lookup)

## Summary

| Task | C# | Python |
|---|---|---|
| Create | `new HashSet<int> { 1, 2 }` | `{1, 2}` |
| Empty | `new HashSet<int>()` | `set()` — **not** `{}` |
| Add | `Add(x)` | `add(x)` |
| Remove (safe) | `Remove(x)` | `discard(x)` |
| Remove (strict) | *(n/a)* | `remove(x)` → `KeyError` |
| Contains | `Contains(x)` | `x in s` |
| Union / intersect | `UnionWith` / `IntersectWith` (mutate) | `a \| b` / `a & b` (new set) |
| Difference | `ExceptWith` | `a - b` |
| Symmetric difference | `SymmetricExceptWith` | `a ^ b` |
| Subset | `IsSubsetOf` | `a <= b` |
| Immutable set | `ImmutableHashSet<T>` / `FrozenSet<T>` | `frozenset` |
| Ordered dedup | `Distinct().ToList()` | `list(dict.fromkeys(xs))` |

## Exercises

1. Given two lists of user names, `week1 = ["alice", "bob", "carol", "bob"]` and `week2 = ["bob", "dave", "carol"]`, find: users active in both weeks, users who churned (week 1 only), and new users (week 2 only).
2. Check whether a sentence is a *pangram* (uses every letter a–z at least once): `"The quick brown fox jumps over the lazy dog"`.
3. Remove duplicates from `[3, 1, 3, 2, 1, 4]` while keeping the original order.
4. Why does `{[1, 2]}` fail but `{(1, 2)}` work?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
week1 = ["alice", "bob", "carol", "bob"]
week2 = ["bob", "dave", "carol"]
w1, w2 = set(week1), set(week2)
both = w1 & w2          # {'bob', 'carol'}
churned = w1 - w2       # {'alice'}
new = w2 - w1           # {'dave'}

# 2
import string
sentence = "The quick brown fox jumps over the lazy dog"
is_pangram = set(string.ascii_lowercase) <= set(sentence.lower())   # True

# 3
result = list(dict.fromkeys([3, 1, 3, 2, 1, 4]))   # [3, 1, 2, 4]

# 4 — set elements must be hashable. Lists are mutable and therefore
#     unhashable; tuples (of hashable items) are immutable and hashable.
```

</details>